# TP4 : Séparations de sources avec un Unet

Binome : 
- Bafou Imad
- Adjal Massyl

# 1. Article

## Petite overview de l'article généré par un LLM :  

### I. Prérequis Académiques

Pour saisir toutes les subtilités de cet article, il faut posséder des bases solides dans trois domaines principaux :

**1. Traitement du Signal Audio (DSP)**
* **Spectrogrammes :** Comprendre ce qu'est un spectrogramme (représentation temps-fréquence) et la différence entre **Magnitude** (amplitude) et **Phase**. L'article repose entièrement sur la manipulation de spectrogrammes de magnitude[cite: 74, 93].
* **Transformée de Fourier à Court Terme (STFT) :** Savoir comment on passe d'un signal audio brut (forme d'onde) à un spectrogramme, et inversement (iSTFT). L'article utilise une fenêtre de 1024 et un saut (hop) de 768 frames[cite: 89].
* **Masquage Temps-Fréquence :** Comprendre le concept de "masque binaire" ou "masque souple" (soft mask) appliqué à un spectrogramme pour filtrer une source spécifique[cite: 43, 44].

**2. Apprentissage Profond (Deep Learning)**
* **Réseaux de Neurones Convolutifs (CNN) :** Comprendre les couches de convolution, le *stride* (pas), le *padding*, et les fonctions d'activation (ReLU, Leaky ReLU)[cite: 85].
* **Architecture Encodeur-Décodeur :** Comprendre comment un réseau compresse l'information (encodage) puis la reconstruit (décodage)[cite: 62].
* **Fonctions de Coût (Loss Functions) :** Notamment la norme L1 et L1,1 utilisée ici pour minimiser l'erreur entre le masque prédit et le masque réel[cite: 77, 78].
* **Surapprentissage et Régularisation :** Comprendre l'utilité du *Dropout* (utilisé à 50% dans ce modèle) et de la *Batch Normalization*[cite: 86].

**3. Mathématiques**
* **Algèbre Linéaire :** Opérations matricielles (multiplication élément par élément, normes).

---

### II. Explication Détaillée de l'Article

L'objectif de l'article est de séparer la **voix chantée** de l'**accompagnement instrumental** dans un enregistrement musical mono[cite: 31]. Les auteurs traitent ce problème comme une tâche de **traduction image-à-image**[cite: 22].

#### 1. L'Architecture : Le U-Net
L'innovation centrale est l'adaptation de l'architecture **U-Net** (créée pour la segmentation d'images biomédicales) à l'audio[cite: 59, 60].


* **Structure en "U" :** Le réseau a une forme d'encodeur (qui réduit la dimension spatiale de l'image pour capturer le contexte global) suivi d'un décodeur (qui augmente la dimension pour retrouver la résolution d'origine)[cite: 62, 63].
* **Le problème des CNN classiques :** Dans les réseaux encodeur-décodeur classiques (type "sablier"), la compression fait perdre les détails fins. Or, en audio, une petite erreur de reconstruction fréquentielle ou temporelle s'entend très distinctement (artefacts)[cite: 65, 66].
* **La solution (Skip Connections) :** Le U-Net ajoute des **connexions de saut** (skip connections) qui relient directement les couches de l'encodeur aux couches correspondantes du décodeur. Cela permet de transférer les informations de bas niveau (détails fins) directement vers la sortie, sans passer par goulot d'étranglement de la compression[cite: 68, 69].

#### 2. La Méthodologie
* **Entrée :** Le spectrogramme de magnitude du mélange audio (Mix).
* **Sortie :** Un "masque" (valeurs entre 0 et 1) prédit par le réseau.
* **Reconstruction :** Ce masque est multiplié par le spectrogramme d'entrée pour isoler la voix (ou l'instrumental). Pour reconstruire le son, ils utilisent la **phase du signal original**, car le réseau ne prédit que la magnitude[cite: 93].
* **Entraînement :** Ils entraînent deux modèles séparés : un pour isoler la voix, un pour isoler les instruments[cite: 72].

#### 3. Le "Hack" du Dataset (Données d'entraînement)
Le manque de données (pistes séparées voix/instruments) est un problème majeur dans ce domaine[cite: 48]. Les auteurs ont contourné ce problème de manière astucieuse :
* Ils ont récupéré des paires de chansons commerciales : la version "Originale" et la version "Instrumentale" officielle[cite: 98, 100].
* En soustrayant l'instrumental de l'original, ils ont obtenu une approximation de la piste vocale[cite: 166].
* Cela leur a permis de créer un dataset massif de **20 000 titres** (environ 2 mois d'audio continu), ce qui était le plus grand à l'époque[cite: 168, 169].

#### 4. Évaluation et Résultats
Ils ont comparé leur modèle U-Net à deux autres modèles : un **modèle de base** (même architecture mais sans les skip connections) et **Chimera** (l'état de l'art de l'époque)[cite: 173, 176].

* **Métriques Objectives :** Ils ont utilisé les mesures standards (SDR, SIR, SAR) sur les datasets publics iKala et MedleyDB. Le U-Net a surpassé les autres modèles sur toutes les métriques[cite: 218].
* **Visualisation :** Ils montrent que le modèle sans skip connections capture la structure globale mais perd les détails fins (spectrogramme flou), alors que le U-Net génère des détails précis[cite: 223].
* **Évaluation Subjective :** Ils ont fait écouter les extraits à des humains via la plateforme CrowdFlower. Les auditeurs ont jugé la qualité sonore et la séparation du U-Net supérieures à celles des autres modèles[cite: 305].

#### 5. Conclusion
L'article conclut que l'architecture U-Net est très efficace pour la séparation de sources grâce à sa capacité à préserver les détails fins via les skip connections, et que l'utilisation d'un grand dataset de qualité commerciale est cruciale pour la généralisation du modèle[cite: 307, 311].


# 2. Données

In [ ]:
import musdb
mus = musdb.DB(download=True)
mus [0]. audio